# ENCODE ChIP-seq QC validation of EpiClass Input-class probability

Claude Opus 4.6 summary

## Purpose

Reviewer response analysis: do EpiClass low-confidence / high Input-class predictions correspond to ChIP-seq experiments that ENCODE flags as poor quality by conventional metrics? Provides external quantitative validation of the manuscript's interpretation that low EpiClass prediction scores and elevated Input-class probabilities reflect noisy ChIP datasets.

## Inputs

1. **ENCODE quality metric objects** (JSON), one list of metric entries per experiment accession, fetched from the ENCODE portal. Multiple metric types per experiment, computed at different pipeline stages.
2. **EpiClass predictions** on ENCODE ChIP-seq files, with per-class softmax probabilities including the Input class.

## Pipeline

### 1. Restrict QC metrics

Restrict ENCODE QC metrics to ChIP-seq experiments present in the EpiClass prediction table.

### 2. Build per-experiment QC table

- Extract a curated set of fields from each metric type (FRiP, JSD, NSC, NRF, PBC1/2, peak counts, library complexity, depth).
- For replicate-level metrics (alignment, library complexity, cross-correlation): aggregate across replicates (mean, with min/max/n). For experiment-level metrics (FRiP, reproducible peaks, histone QC): keep only the pooled entry by matching biological_replicates to the full replicate set for the experiment.
- Coalesce metrics that appear in multiple metric types (e.g. NRF in both `ChipLibraryQualityMetric` and the older `ChipSeqFilterQualityMetric`) into single consensus columns, preferring the more populated source.
- Tag pipeline version (new vs legacy) from which metric types are present; drop the small legacy subset to avoid batch effects from non-comparable NSC/RSC computations across pipeline versions.

### 3. Filter EpiClass predictions

- **Exclude Input control experiments.** The reviewer's hypothesis is about ChIP samples, not controls; including Inputs would inflate correlations trivially since real Inputs both have low FRiP/NSC and high Input-class probability.
- **Keep only experiments with a single signal bigwig file**, so that the QC metrics unambiguously correspond to the file EpiClass scored.

### 4. Join

Join QC table to filtered predictions on `EXPERIMENT_accession`.

### 5. Per-target Spearman correlations

Compute Spearman correlations between Input-class probability and four QC metrics (FRiP, reproducible peaks, JSD, NSC), separately per target (per histone mark or per TF/chromatin regulator). Targets with fewer than 10 experiments are excluded. P-values are computed via permutation tests (9,999 resamples, two-sided, permutation of pairings).

> **Note:** max-prediction-score correlations were tried first but are unusable for histones due to ceiling effects (>90% of experiments score >0.99). Input-class probability avoids this saturation because it has meaningful variance even for confidently-predicted samples.

### 6. Restrict histone analysis to experiments with non-trivial uncertainty

EpiClass classifies the vast majority of ENCODE histone experiments with near-perfect confidence (Input-class probability typically < 10⁻⁵). In this regime, the model's uncertainty is too low for variation in Input-class probability to track meaningful differences in sample quality. When all histone experiments are included, within-mark correlations between Input probability and QC metrics are near zero — not because the relationship doesn't exist, but because there is insufficient variance in the predictor.

Restricting to experiments with max prediction score < 0.99 isolates the subset where the model assigns non-trivial uncertainty. H3K4me3 is excluded from this analysis due to insufficient sample size after applying the filter.

### 7. Plot

Two-panel boxplot figure: core histone marks (prediction score < 0.99) and non-core targets (TFs, chromatin regulators, Pol II subunits). Each point is one target/mark; boxplots summarize the per-target distribution of Spearman ρ values. Histone marks are individually colored; non-core targets shown as small black dots. Negative ρ indicates the reviewer-predicted direction.

## Key findings

- **Non-core targets (TFs, chromatin regulators, Pol II):** per-target correlations are predominantly negative across all four QC metrics (FRiP, reproducible peaks, JSD, NSC). Directly supports the reviewer's interpretation.
- **Core histone marks:** when restricted to experiments with prediction score < 0.99, per-mark correlations are predominantly negative for narrow histone marks. For broad marks, the correlations are weaker and not always significant, consistent with the known reduced sensitivity of conventional QC metrics for broad chromatin marks (Landt et al. 2012; Nakato & Sakata 2021).
- **Ceiling effect, not absence of signal:** the lack of correlation when all histone experiments are included reflects the high confidence of EpiClass on the ENCODE histone corpus, not a failure of the quality signal.


## SETUP

In [ ]:
# pylint: disable=import-error, redefined-outer-name, too-many-lines, use-dict-literal, missing-module-docstring, pointless-statement, too-many-branches, nested-min-max
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from scipy.stats import rankdata, spearmanr

from epiclass.utils.notebooks.paper.paper_utilities import (
    ASSAY_ORDER,
    IHECColorMap,
    save_figure,
)

In [ ]:
base_dir = Path.home() / "Projects/epiclass/output/paper"

base_fig_dir = base_dir / "figures"

metadata_dir = base_dir / "data/metadata"

encode_metadata_dir = metadata_dir / "encode"

preds_dir = base_dir / "data" / "training_results"

for path in [metadata_dir, encode_metadata_dir, preds_dir, base_fig_dir]:
    if not path.exists():
        raise ValueError(f"Path {path} does not exist.")

In [ ]:
IHECColorMap = IHECColorMap(base_fig_dir)
assay_colors = IHECColorMap.assay_color_map

In [ ]:
qc_metrics_path = (
    encode_metadata_dir
    / "encode_experiment_quality_metrics_2025-02_no_revoked_hg38.freeze1.json"
)
with open(qc_metrics_path, "r", encoding="utf8") as f:
    qc_metrics = json.load(f)
print(f"QC metrics N: {len(qc_metrics)}")

In [ ]:
filepath = (
    preds_dir
    / "dfreeze_v2/predictions/encode/complete_encode_predictions_augmented_2025-02_metadata.csv.gz"
)
encode_preds = pd.read_csv(filepath, low_memory=False)
display(encode_preds.shape)

## Acquire ChIP QC metrics

In [ ]:
preds_chip = encode_preds[
    encode_preds["FILE_assay_title"].str.contains("chip", case=False, na=False)
]
print(f"ChIP-seq metadata shape: {preds_chip.shape}")

In [ ]:
exp_id_label = "EXPERIMENT_accession"
chip_exp_acc = preds_chip[exp_id_label].unique()
print(f"Number of unique ChIP-seq experiments: {len(chip_exp_acc)}")

qc_metrics = {k: v for k, v in qc_metrics.items() if k in chip_exp_acc}
print(f"QC metrics N after filtering: {len(qc_metrics)}")

In [ ]:
# Sanity check: Ensure that all experiments in qc_metrics have the 'quality_metric' key
for acc, metrics_list in qc_metrics.items():
    for metrics in metrics_list:
        if "quality_metric" not in metrics:
            print(f"Missing 'quality_metric' key for experiment {acc}")

In [ ]:
metrics_type_count = Counter()
metrics_type_entries = defaultdict(set)
for acc in qc_metrics:
    all_metrics = qc_metrics[acc]
    # print(f"Experiment {acc} has {len(all_metrics)} quality metrics entries.")
    for metrics in all_metrics:
        details = metrics["quality_metric"]

        # available
        metrics_type_count.update([details["@type"][0]])

        # track which keys are present for each metric type
        metrics_type_entries[details["@type"][0]].update(details.keys())

In [ ]:
display(metrics_type_count.most_common())

display(metrics_type_entries)

In [ ]:
FIELDS_BY_TYPE = {
    "ChipPeakEnrichmentQualityMetric": ["frip"],
    "ChipReplicationQualityMetric": ["reproducible_peaks"],
    "ChipAlignmentEnrichmentQualityMetric": [
        "jsd",
        "NSC",
        "RSC",
        "pct_genome_enrich",
        "diff_enrich",
    ],
    "ChipLibraryQualityMetric": ["NRF", "PBC1", "PBC2"],
    "ChipSeqFilterQualityMetric": ["NRF", "PBC1", "PBC2", "NSC", "RSC"],
    "ChipAlignmentQualityMetric": [
        "usable_fragments",
        "mapped_reads",
        "pct_mapped_reads",
    ],
    "HistoneChipSeqQualityMetric": ["frip", "npeak_overlap", "nreads_in_peaks"],
}

In [ ]:
def collect_experiment_metrics(qc_metrics):
    """
    Build a per-experiment QC table from ENCODE quality metric objects.

    ENCODE provides multiple QC metric entries per experiment, computed at
    different pipeline stages and on different files. This function extracts
    a curated set of fields (defined by the module-level FIELDS_BY_TYPE dict),
    handles the distinction between replicate-level and experiment-level
    metrics, and aggregates into one row per experiment.

    Metric level distinction
    ------------------------
    ENCODE QC metrics fall into two categories:

    **Replicate-level** (ChipAlignmentQualityMetric, ChipAlignmentEnrichment-
    QualityMetric, ChipLibraryQualityMetric, ChipSeqFilterQualityMetric):
    computed per replicate BAM file. An experiment with 3 replicates will
    have 3 entries for each of these metric types. These are averaged across
    replicates, which is standard practice — the final bigwig is derived
    from all replicates, so the mean replicate-level QC characterizes the
    pooled output.

    **Experiment-level** (ChipPeakEnrichmentQualityMetric, ChipReplication-
    QualityMetric, HistoneChipSeqQualityMetric): computed on pooled data
    (peak calls from merged replicates). However, ENCODE also stores
    per-replicate peak metrics (e.g., FRiP for each replicate's peaks),
    and experiments reprocessed by multiple pipeline versions have separate
    pooled entries for each version. To avoid mixing per-replicate with
    pooled values, this function keeps only entries whose biological_replicates
    field matches the full set of replicates for the experiment (i.e., the
    pooled entry). For single-replicate experiments, all entries have
    biological_replicates=[1], so they pass through correctly. When multiple
    pipeline versions both produce pooled entries, their values are averaged;
    the spread is typically negligible.

    Aggregation
    -----------
    For each (metric_type, field) pair within an experiment, the function
    stores:
    - ``__mean``: mean across all qualifying entries
    - ``__n``: count of non-NaN entries (useful for diagnosing unexpected
      averaging — n=1 for experiment-level metrics in single-pipeline
      experiments, n=2-3 for replicate-level metrics)
    - ``__min``, ``__max``: range, only when n > 1, for verifying that
      averaged values are consistent

    Parameters
    ----------
    qc_metrics : dict[str, list[dict]]
        Mapping from experiment accession to a list of metric entries as
        returned by the ENCODE API. Each entry is a dict with keys:
        - 'quality_metric': dict containing '@type' (list of type strings),
          metric field names and values, and 'quality_metric_of' (list of
          file paths this metric applies to).
        - 'biological_replicates': list of int, indicating which biological
          replicates this metric covers (e.g. [1] for replicate 1, [1,2,3]
          for the pooled analysis across all replicates).
        - 'files': list of file paths this entry is associated with.

    Returns
    -------
    pd.DataFrame
        One row per experiment, indexed by experiment accession. Columns
        follow the naming convention ``<MetricType>__<field>__<stat>``
        where stat is one of mean, n, min, max. Only experiments with at
        least one extractable metric value are included.

    See Also
    --------
    FIELDS_BY_TYPE : module-level dict defining which fields to extract
        from each metric type.
    add_consensus_columns : coalesces the per-metric-type columns into
        short-named consensus columns (e.g. 'frip', 'NSC', 'NRF').
    """
    # raw[acc][(type, field)] = list of numeric values across qualifying entries
    raw = defaultdict(lambda: defaultdict(list))

    # Experiment-level metric types: these have separate per-replicate and
    # pooled entries, and we only want the pooled ones.
    EXPERIMENT_LEVEL_TYPES = {
        "ChipPeakEnrichmentQualityMetric",
        "ChipReplicationQualityMetric",
        "HistoneChipSeqQualityMetric",
    }

    for acc, entries in qc_metrics.items():
        # First pass: determine the full set of biological replicates for
        # this experiment by taking the union across all entries. This is
        # used to identify pooled entries (whose bio_reps == all_reps).
        all_reps = set()
        for entry in entries:
            all_reps.update(entry.get("biological_replicates", []))

        # Second pass: extract metric values
        for entry in entries:
            details = entry["quality_metric"]
            mtype = details["@type"][0]

            # Skip metric types we don't need
            if mtype not in FIELDS_BY_TYPE:
                continue

            bio_reps = entry.get("biological_replicates", [])

            # For experiment-level metrics, skip per-replicate entries and
            # keep only pooled entries (bio_reps matches the full replicate
            # set). For single-replicate experiments, bio_reps=[1] matches
            # all_reps={1}, so the entry passes through.
            if mtype in EXPERIMENT_LEVEL_TYPES and set(bio_reps) != all_reps:
                continue

            # Extract each field defined for this metric type
            for field in FIELDS_BY_TYPE[mtype]:
                val = details.get(field)
                if val is None:
                    continue
                try:
                    raw[acc][(mtype, field)].append(float(val))
                except (TypeError, ValueError):
                    # Skip non-numeric values (e.g. plot objects, strings)
                    continue

    # Build the output DataFrame: one row per experiment, with summary
    # statistics for each (metric_type, field) pair
    rows = []
    for acc, metric_dict in raw.items():
        row = {"experiment": acc}
        for (mtype, field), values in metric_dict.items():
            arr = np.asarray(values, dtype=float)
            prefix = f"{mtype}__{field}"
            row[f"{prefix}__mean"] = float(np.nanmean(arr))
            row[f"{prefix}__n"] = int(np.sum(~np.isnan(arr)))
            if arr.size > 1:
                row[f"{prefix}__min"] = float(np.nanmin(arr))
                row[f"{prefix}__max"] = float(np.nanmax(arr))
        rows.append(row)

    df = pd.DataFrame(rows).set_index("experiment")
    return df

In [ ]:
def add_consensus_columns(df):
    """
    Add consensus QC metric columns by coalescing across metric type sources.

    Several ENCODE QC metrics appear under multiple metric types because the
    ENCODE pipeline has been versioned over time and because the histone and
    TF pipelines emit different metric objects. For example, NRF/PBC1/PBC2/
    NSC/RSC are present in the legacy `ChipSeqFilterQualityMetric` and were
    later split into `ChipLibraryQualityMetric` (complexity) and
    `ChipAlignmentEnrichmentQualityMetric` (cross-correlation) in the newer
    pipeline; FRiP appears in `ChipPeakEnrichmentQualityMetric` (uniform
    source) as well as in the pipeline-specific `HistoneChipSeqQualityMetric`
    and `IDRQualityMetric` summaries.

    This function creates a single short-named column per metric (e.g. `frip`,
    `NSC`, `NRF`) by coalescing across the relevant source columns produced
    by `collect_experiment_metrics`, preferring the more uniformly populated
    source and falling back to pipeline-specific sources as needed.

    Peak counts are deliberately NOT coalesced into a single cross-pipeline
    column because the histone pipeline (`npeak_overlap`, overlap-based) and
    the TF pipeline (`N_optimal`, IDR-based) define them differently. Two
    separate columns `peaks_histone` and `peaks_tf` are created instead, plus
    a `peaks_any` convenience column intended only for within-class analyses.

    Parameters
    ----------
    df : pd.DataFrame
        Per-experiment QC table from `collect_experiment_metrics`, indexed by
        experiment accession, with columns named `<metric_type>__<field>__mean`.

    Returns
    -------
    pd.DataFrame
        The input DataFrame with the following consensus columns added:
        `frip`, `peaks_histone`, `peaks_tf`, `peaks_any`, `jsd`, `NSC`, `RSC`,
        `NRF`, `PBC1`, `PBC2`, `pct_genome_enrich`, `reproducible_peaks`,
        `usable_fragments`. Any consensus column whose source columns are all
        absent will be filled with NaN.
    """

    def coalesce(*cols):
        """
        Combine multiple columns into a single Series, preferring earlier
        columns and filling missing values from later ones.

        For each row, takes the value from the first column in `cols` that
        exists in `df` and is non-null; if that value is null, falls through
        to the next column, and so on. Columns listed in `cols` that do not
        exist in `df` are silently skipped, allowing the function to be
        called with a superset of possible source columns. If none of the
        listed columns exist in `df`, returns an all-NaN Series with the
        same index.

        Parameters
        ----------
        *cols : str
            Column names to coalesce, in order of preference (most preferred
            first).

        Returns
        -------
        pd.Series
            Coalesced values aligned to `df.index`.
        """
        existing = [c for c in cols if c in df.columns]
        if not existing:
            return pd.Series(np.nan, index=df.index)
        out = df[existing[0]].copy()
        for c in existing[1:]:
            out = out.fillna(df[c])
        return out

    # FRiP: prefer the uniform source, fall back to pipeline-specific
    df["frip"] = coalesce(
        "ChipPeakEnrichmentQualityMetric__frip__mean",
        "HistoneChipSeqQualityMetric__frip__mean",
        "IDRQualityMetric__frip__mean",
    )

    # Peak count: keep histone and TF separate
    df["peaks_histone"] = coalesce("HistoneChipSeqQualityMetric__npeak_overlap__mean")
    df["peaks_tf"] = coalesce("IDRQualityMetric__N_optimal__mean")
    # Optional unified column for within-class analyses only — DO NOT pool across classes
    df["peaks_any"] = df["peaks_histone"].fillna(df["peaks_tf"])

    # Cross-pipeline-safe metrics
    df["jsd"] = coalesce("ChipAlignmentEnrichmentQualityMetric__jsd__mean")
    df["NSC"] = coalesce(
        "ChipAlignmentEnrichmentQualityMetric__NSC__mean",
        "ChipSeqFilterQualityMetric__NSC__mean",
    )
    df["RSC"] = coalesce(
        "ChipAlignmentEnrichmentQualityMetric__RSC__mean",
        "ChipSeqFilterQualityMetric__RSC__mean",
    )
    df["NRF"] = coalesce(
        "ChipLibraryQualityMetric__NRF__mean",
        "ChipSeqFilterQualityMetric__NRF__mean",
    )
    df["PBC1"] = coalesce(
        "ChipLibraryQualityMetric__PBC1__mean",
        "ChipSeqFilterQualityMetric__PBC1__mean",
    )
    df["PBC2"] = coalesce(
        "ChipLibraryQualityMetric__PBC2__mean",
        "ChipSeqFilterQualityMetric__PBC2__mean",
    )
    df["pct_genome_enrich"] = coalesce(
        "ChipAlignmentEnrichmentQualityMetric__pct_genome_enrich__mean"
    )
    df["reproducible_peaks"] = coalesce(
        "ChipReplicationQualityMetric__reproducible_peaks__mean"
    )
    df["usable_fragments"] = coalesce(
        "ChipAlignmentQualityMetric__usable_fragments__mean"
    )
    return df

In [ ]:
df_full = collect_experiment_metrics(qc_metrics)
print(f"Full QC metrics table shape: {df_full.shape}")

In [ ]:
df_full = add_consensus_columns(df_full)

#### Pipeline versions filtering

`ChipSeqFilterQualityMetric` vs `ChipLibraryQualityMetric` + `ChipAlignmentEnrichmentQualityMetric`

These are from different pipeline versions. ChipSeqFilterQualityMetric is the older combined metric (NRF, PBC1, PBC2, NSC, RSC all in one object) from the legacy pipeline; the newer pipeline split these into `ChipLibraryQualityMetric` (complexity) and `ChipAlignmentEnrichmentQualityMetric` (cross-correlation). Your current `coalesce` handles this correctly by falling back, but be aware that mixing experiments processed by different pipeline versions introduces a batch effect. The newer pipeline's NSC/RSC values aren't perfectly comparable to the older one's because the underlying tools and subsampling differ.

We exclude legacy pipeline metrics because they're only associated with n=10 files.

In [ ]:
df_full["pipeline_version"] = np.where(
    df_full["ChipLibraryQualityMetric__NRF__mean"].notna(),
    "new",
    np.where(
        df_full["ChipSeqFilterQualityMetric__NRF__mean"].notna(), "legacy", "unknown"
    ),
)

In [ ]:
df_full["pipeline_version"].value_counts(dropna=False)

Only 10 legacy pipeline datasets, excluding them.

In [ ]:
df_full = df_full[df_full["pipeline_version"] == "new"]
display(df_full.shape)

In [ ]:
# Compact table
key_cols = [
    "frip",
    "reproducible_peaks",
    "jsd",
    "NSC",
    "RSC",
    "NRF",
    "PBC1",
    "PBC2",
    "pct_genome_enrich",
    "usable_fragments",
]
df_key = df_full[key_cols]

print(f"Experiments with data: {len(df_key)}")
print(f"Coverage per metric:\n{df_key.notna().sum()}")

We're focusing on the effect of the input prediction score for non-control experiments, so we exclude input files.

In [ ]:
# excluding input
encode_preds = encode_preds[
    encode_preds["assay_epiclass"].str.contains("h3*|core", case=False, regex=True)
]
print(f"ChIP-seq files after input filtering: {encode_preds.shape[0]}")

We also exclude experiments with more than 1 file, since we can't be sure which file the QC metrics correspond to.

In [ ]:
groupby_experiment = encode_preds.groupby("EXPERIMENT_accession")
print(f"Number of experiments: {len(groupby_experiment)}")
files_per_exp = groupby_experiment["FILE_accession"].nunique()

N_all = encode_preds.shape[0]

single_file_exps = files_per_exp[files_per_exp == 1].index
N_single = len(single_file_exps)
print(f"Experiments with single file: {N_single}")
print(f"Fraction with single file: {N_single / N_all:.2%}")
print(f"Experiments with multiple files: {len(files_per_exp) - N_single}")

In [ ]:
single_file_preds = encode_preds[
    encode_preds["EXPERIMENT_accession"].isin(single_file_exps)
].copy()
print(f"Predictions for single-file experiments: {single_file_preds.shape}")

In [ ]:
files_per_exp[files_per_exp != 1]

Analysis will treat core/non-core separately, so we tag them

In [ ]:
# h3* are core, the rest are non-core
single_file_preds.loc[:, "experiment_group"] = single_file_preds.loc[
    :, "assay_epiclass"
].apply(lambda x: "core" if "h3" in x.lower() else "non-core")

In [ ]:
single_file_preds["experiment_group"].value_counts()

## Spearman correlation analysis

In [ ]:
INPUT_COL = "input (assay_epiclass_7c)"
SCORE_COL = "Max pred (assay_epiclass_7c)"

In [ ]:
# Log-transform Input probability (it spans orders of magnitude)
# Add a small floor to avoid log(0)
single_file_preds.loc[:, "log_input"] = np.log10(
    single_file_preds.loc[:, INPUT_COL].clip(lower=1e-10)
)

In [ ]:
merged = pd.DataFrame.join(
    single_file_preds.set_index("EXPERIMENT_accession"),
    df_key,
    how="inner",
)
print(f"Merged table shape: {merged.shape}")

core = merged[merged["experiment_group"] == "core"].copy()
print(f"Core files: {len(core)}")
non_core = merged[merged["experiment_group"] == "non-core"].copy()
print(f"Non-core files: {len(non_core)}")

In [ ]:
print("Input probability distribution:")
for name, df in [("core", core), ("non-core", non_core)]:
    print(f"\n{name} (n={len(df)}):")
    print(df[INPUT_COL].describe())
    print(f"  log10 range: [{df['log_input'].min():.2f}, {df['log_input'].max():.2f}]")

In [ ]:
def correlate_input_with_qc(
    merged,
    qc_cols,
    target_col="assay",
    score_col="log_input",
    min_n=20,
):
    """
    Spearman correlations between Input-class probability and QC metrics,
    pooled and stratified by target.
    """
    # Pooled within group
    pooled = []
    for col in qc_cols:
        sub = merged[[score_col, col]].dropna()
        if len(sub) < min_n:
            continue
        rho, p = spearmanr(sub[score_col], sub[col])
        pooled.append({"metric": col, "spearman_rho": rho, "p_value": p, "n": len(sub)})
    pooled = pd.DataFrame(pooled)

    # Within target
    within = []
    for target, tg in merged.groupby(target_col):
        if len(tg) < min_n:
            continue
        for col in qc_cols:
            sub = tg[[score_col, col]].dropna()
            if len(sub) < min_n:
                continue
            rho, p = spearmanr(sub[score_col], sub[col])
            within.append(
                {
                    "target": target,
                    "metric": col,
                    "spearman_rho": rho,
                    "p_value": p,
                    "n": len(sub),
                }
            )
    within = pd.DataFrame(within)

    # Fisher-z meta-analysis
    if len(within):

        def fisher_mean(g):
            """Compute the weighted mean of Spearman rhos using Fisher z-transform."""
            z = np.arctanh(g["spearman_rho"].clip(-0.999, 0.999))
            w = g["n"] - 3
            z_bar = np.average(z, weights=w)
            return pd.Series(
                {
                    "spearman_rho_meta": np.tanh(z_bar),
                    "n_targets": len(g),
                    "n_total": int(g["n"].sum()),
                }
            )

        meta = (
            within.groupby("metric", group_keys=False)  # type: ignore
            .apply(fisher_mean, include_groups=False)  # type: ignore
            .reset_index()
        )
    else:
        meta = pd.DataFrame()

    return pooled, within, meta

### Graph

In [ ]:
# %% [markdown]
# ### Graphing utilities

# %%
PLOT_METRICS = [
    "frip",
    "reproducible_peaks",
    "jsd",
    "NSC",
    # "RSC", # less reliable generally
]
METRIC_LABELS = {
    "frip": "FRiP",
    "reproducible_peaks": "Repro.<br>peaks",
    "jsd": "JSD",
    "NSC": "NSC",
    # "RSC": "RSC",
}
PANEL_COLORS = [
    "rgba(31, 119, 180, 0.95)",  # blue
    "rgba(44, 160, 44, 0.95)",  # green
    "rgba(214, 39, 40, 0.95)",  # red
    "rgba(148, 103, 189, 0.95)",  # purple
    "rgba(255, 127, 14, 0.95)",  # orange
]

In [ ]:
qc_cols = PLOT_METRICS

Boxplot version

In [ ]:
def spearman_permutation_test(x, y, n_permutations=9999, rng=None):
    """
    Spearman rank correlation with vectorized permutation p-value.

    Computes the Spearman correlation between x and y, then estimates a
    two-sided p-value by permuting the pairing between x and y and
    recomputing the correlation for each permutation. The null distribution
    is computed efficiently via Pearson correlation on ranks using matrix
    operations rather than repeated calls to spearmanr.

    Parameters
    ----------
    x, y : array-like
        Input arrays of the same length. NaNs should be removed before calling.
    n_permutations : int
        Number of random permutations for the null distribution.
    rng : np.random.Generator or None
        Random number generator. If None, creates one with default seed.

    Returns
    -------
    dict with keys:
        - spearman_rho : float
        - p_value : float (two-sided permutation p-value)
        - n : int (number of observations)
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) != len(y):
        raise ValueError("x and y must have the same length")
    if len(x) < 3:
        return {"spearman_rho": np.nan, "p_value": np.nan, "n": len(x)}

    if rng is None:
        rng = np.random.default_rng()

    n = len(x)
    rho = spearmanr(x, y).statistic  # type: ignore

    # Rank using scipy (handles ties by averaging, consistent with spearmanr)
    rank_x = rankdata(x)
    rank_y = rankdata(y)

    # Center y-ranks once
    rank_y_centered = rank_y - rank_y.mean()
    rank_y_ss = (rank_y_centered**2).sum()

    # Generate all permutations of x-ranks at once: (n_permutations, n)
    perms = rng.permuted(np.tile(rank_x, (n_permutations, 1)), axis=1)
    perms_centered = perms - perms.mean(axis=1, keepdims=True)

    # Vectorized Pearson on ranks = Spearman
    num = perms_centered @ rank_y_centered
    denom = np.sqrt((perms_centered**2).sum(axis=1) * rank_y_ss)

    # Guard against zero denominator (constant input)
    with np.errstate(divide="ignore", invalid="ignore"):
        null_rhos = np.where(denom > 0, num / denom, 0.0)

    # Two-sided p-value with standard +1 correction
    p_value = (np.sum(np.abs(null_rhos) >= np.abs(rho)) + 1) / (n_permutations + 1)

    return {"spearman_rho": rho, "p_value": p_value, "n": n}

In [ ]:
def compute_per_target_correlations(
    merged,
    qc_cols,
    target_col="assay",
    score_col="log_input",
    min_n=10,
    n_permutations=9999,
    seed=42,
):
    """
    Compute one Spearman rho per (target, metric) pair, with permutation
    p-values.
    """
    rng = np.random.default_rng(seed)

    rows = []
    for target, tg in merged.groupby(target_col):
        if len(tg) < min_n:
            continue
        for col in qc_cols:
            sub = tg[[score_col, col]].dropna()
            if len(sub) < min_n:
                continue

            result = spearman_permutation_test(
                sub[score_col].values,
                sub[col].values,
                n_permutations=n_permutations,
                rng=rng,
            )
            result["target"] = target
            result["metric"] = col
            rows.append(result)

    return pd.DataFrame(rows)

In [ ]:
def plot_qc_correlation_boxplots(
    panels,
    metrics=None,
    metric_labels=None,
    title="EpiClass Input-class probability vs ENCODE ChIP-seq QC metrics",
    width=None,
    height=550,
    jitter_width=0.15,
    logdir=None,
):
    """
    Two-panel figure with boxplots of per-target Spearman correlations.

    Parameters
    ----------
    panels : dict[str, dict]
        Keys are panel labels (e.g. "A", "B").
        Values are dicts with keys:
            - "title": str, panel subtitle
            - "subtitle": str, e.g. "17 targets"
            - "per_target_df": pd.DataFrame from compute_per_target_correlations
            - "color": str (optional)
            - "show_points": bool (optional, default False). If True, overlay
              individually colored target points with legend entries.
            - "show_all_points": bool (optional, default False). If True,
              overlay all points as small black dots (no legend entries).
    metrics : list[str], optional
    metric_labels : dict[str, str], optional
    title : str
    width : int, optional
    height : int
    jitter_width : float
        Max horizontal displacement for points (in axis units). Each target
        gets a deterministic offset so it's consistent across metrics.
    logdir : str or Path, optional

    Returns
    -------
    go.Figure
    """
    if metrics is None:
        metrics = PLOT_METRICS
    if metric_labels is None:
        metric_labels = METRIC_LABELS

    x_labels = [metric_labels.get(m, m) for m in metrics]
    n_panels = len(panels)
    if width is None:
        width = 500 * n_panels

    panel_list = list(panels.items())

    fig = make_subplots(
        rows=1,
        cols=n_panels,
        subplot_titles=[
            f"<b>{label}.</b> {info['title']}<br>"
            f"<span style='font-size:14px'>{info['subtitle']}</span>"
            for label, info in panel_list
        ],
        horizontal_spacing=0.12,
        shared_yaxes=True,
    )

    all_rhos = []
    for idx, (label, info) in enumerate(panel_list):
        col = idx + 1
        color = info.get("color", PANEL_COLORS[idx % len(PANEL_COLORS)])
        show_points = info.get("show_points", False)
        show_all_points = info.get("show_all_points", False)
        per_target = info["per_target_df"]

        # Build deterministic jitter offsets: one per target, evenly spaced
        all_targets = sorted(per_target["target"].unique())
        if show_points:
            targets = [t for t in ASSAY_ORDER if t in per_target["target"].values]
        else:
            targets = all_targets

        n_targets = len(all_targets)
        if n_targets > 1:
            offsets = np.linspace(-jitter_width, jitter_width, n_targets)
        else:
            offsets = [0.0]
        target_offset = dict(zip(all_targets, offsets))

        for i, metric in enumerate(metrics):
            metric_data = per_target[per_target["metric"] == metric]
            rhos = metric_data["spearman_rho"].tolist()
            all_rhos.extend(rhos)

            # Boxplot at numeric x position
            points = "all" if show_all_points else "outliers"

            fig.add_trace(
                go.Box(
                    y=rhos,
                    x0=i,
                    name=x_labels[i],
                    marker_color=color,
                    line_color=color,
                    fillcolor=color.replace("0.95", "0.3"),
                    showlegend=False,
                    boxmean=True,
                    whiskerwidth=0.5,
                    width=0.5,
                    boxpoints=points,
                    marker=dict(size=4, line=dict(width=0.5, color="black")),
                ),
                row=1,
                col=col,
            )

            # Colored points with legend entries
            if show_points:
                for target in targets:
                    row_data = metric_data[metric_data["target"] == target]
                    if row_data.empty:
                        continue
                    rho = row_data["spearman_rho"].values[0]
                    n = row_data["n"].values[0]
                    x_pos = i + target_offset[target]

                    fig.add_trace(
                        go.Scatter(
                            y=[rho],
                            x=[x_pos],
                            mode="markers",
                            marker=dict(
                                color=assay_colors[target],
                                size=8,
                                line=dict(width=1, color="black"),
                            ),
                            showlegend=(i == 0),
                            legendgroup=target,
                            name=f"{target.lower()} (n={n})",
                            hovertemplate=(
                                f"{target.lower()}<br>"
                                f"ρ={rho:.3f}<br>"
                                f"n={n}<extra></extra>"
                            ),
                        ),
                        row=1,
                        col=col,
                    )

        fig.add_hline(
            y=0, line_width=1, line_color="black", line_dash="dash", row=1, col=col  # type: ignore
        )

        # Set x-axis to use numeric positions with custom tick labels
        fig.update_xaxes(
            tickvals=list(range(len(metrics))),
            ticktext=x_labels,
            tickangle=-30,
            range=[-0.6, len(metrics) - 0.4],
            row=1,
            col=col,
        )

    # Y axis
    y_min = -1
    y_max = 1

    for col in range(1, n_panels + 1):
        fig.update_yaxes(
            range=[y_min, y_max],
            zeroline=False,
            gridcolor="rgba(0,0,0,0.08)",
            row=1,
            col=col,
            dtick=0.2,
        )

    fig.update_yaxes(
        title_text="Spearman ρ (Input prob. vs QC metric)",
        row=1,
        col=1,
    )

    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center"),
        plot_bgcolor="white",
        width=width,
        height=height,
        margin=dict(t=110, b=80, l=70, r=30),
    )

    if logdir is not None:
        save_figure(
            fig, logdir=logdir, filename="encode_qc_correlation_boxplots", scale=2
        )

    fig.show()
    return fig

In [ ]:
# Compute per-target correlations
max_predScore = 0.99

print(f"Total non-core files: {len(non_core)}")
non_core_subset = non_core[non_core[SCORE_COL] < max_predScore]
print(f"Non-core files with {SCORE_COL} < {max_predScore}: {len(non_core_subset)}")
non_core_per_target = compute_per_target_correlations(non_core_subset, qc_cols, min_n=10)
n_non_core = non_core_per_target[non_core_per_target["metric"] == "frip"]["n"].sum()
print(f"Total non-core files retained: {n_non_core}")

print(f"\nTotal core files: {len(core)}")
core_subset = core[core[SCORE_COL] < max_predScore]
print(f"Core files with {SCORE_COL} < {max_predScore}: {len(core_subset)}")
core_per_target = compute_per_target_correlations(core_subset, qc_cols, min_n=10)
n_core = core_per_target[core_per_target["metric"] == "frip"]["n"].sum()
print(f"Total core files retained: {n_core}")

n_non_core_targets = non_core_per_target["target"].nunique()
n_core_marks = core_per_target["target"].nunique()

In [ ]:
remerged_df = pd.concat([non_core_subset, core_subset], ignore_index=True)
print(f"\nRe-merged subset shape: {remerged_df.shape}")

In [ ]:
core_subset["assay"].value_counts()

In [ ]:
panels = {
    "A": {
        "title": f"Core histone marks (N={n_core})",
        "subtitle": f"{n_core_marks} targets, one point per mark",
        "per_target_df": core_per_target,
        "color": "rgba(214, 39, 40, 0.95)",
        "show_points": True,
    },
    "B": {
        "title": f"Non-core targets (N={n_non_core})",
        "subtitle": f"{n_non_core_targets} targets, one point per target",
        "per_target_df": non_core_per_target,
        "color": "rgba(31, 119, 180, 0.95)",
        "show_all_points": True,
    },
}

fig = plot_qc_correlation_boxplots(
    panels,
    height=500,
    title=f"EpiClass Assay classifier (7c) Input-class probability vs ENCODE ChIP-seq QC metrics (predScore < {max_predScore:.2f})",
    logdir=Path.home() / "downloads",
)

In [ ]:
dfs = []
for spearman_table in [non_core_per_target, core_per_target]:
    table_n = spearman_table.groupby("target")["n"].first()

    display_df = pd.DataFrame(index=table_n.index)
    display_df["n"] = table_n
    for metric in PLOT_METRICS:
        metric_data = spearman_table[spearman_table["metric"] == metric].set_index(
            "target"
        )
        display_df[f"{metric}_rho"] = metric_data["spearman_rho"].round(3)
        display_df[f"{metric}_p"] = metric_data["p_value"]

    # print(display_df.to_string())
    dfs.append(display_df)

In [ ]:
all_results = pd.concat(dfs, keys=["non-core", "core"])
all_results["group"] = all_results.index.get_level_values(0)
all_results["target"] = all_results.index.get_level_values(1)
all_results.set_index(["target"], inplace=True)

In [ ]:
all_results.to_csv(
    Path.home() / "downloads" / "per_target_qc_correlations_7c_predScore0.99.csv"
)

## More complex treatment - Archive

In [ ]:
def classify_repro_entry(details):
    """
    Classify a ChipReplicationQualityMetric entry as 'optimal',
    'conservative', or 'unknown' based on available metadata.

    For IDR pipeline (TF ChIP-seq): uses idr_dispersion_plot filename.
    For overlap pipeline (histone ChIP-seq): uses presence of
    self_consistency_ratio/rescue_ratio fields (present on optimal only).
    """
    # IDR pipeline: dispersion plot filename
    idr_plot = details.get("idr_dispersion_plot", {})
    fn = idr_plot.get("download", "") if isinstance(idr_plot, dict) else ""
    if "pr1_vs" in fn and "pr2" in fn:
        return "optimal"
    if re.match(r"^rep\d+_vs_rep\d+", fn):
        return "conservative"

    # Overlap pipeline: rescue_ratio present on optimal entry only
    if details.get("rescue_ratio") is not None:
        return "optimal"
    # Has assay_term_name but no rescue_ratio → conservative overlap entry
    if details.get("assay_term_name") is not None:
        return "conservative"

    return "unknown"


def select_reproducible_peaks(entries, all_reps):
    """
    From pooled ChipReplicationQualityMetric entries for one experiment,
    select a single reproducible_peaks value.

    Uses max(reproducible_peaks) across all qualifying pooled entries.
    This equals the optimal peak count in ~91% of experiments and gives
    the benefit of the doubt to experiments with IDR edge cases (where
    conservative > optimal due to noisy pseudoreplicate concordance).

    Also returns the classified type for auditing.

    Parameters
    ----------
    entries : list[dict]
        All QC metric entries for this experiment.
    all_reps : set
        Full set of biological replicate numbers for this experiment.

    Returns
    -------
    peaks : float or None
    peak_type : str or None
        'optimal_idr', 'optimal_overlap', 'max_fallback', or None.
    n_candidates : int
        Number of qualifying pooled entries considered.
    """
    candidates = []
    for e in entries:
        details = e["quality_metric"]
        if details["@type"][0] != "ChipReplicationQualityMetric":
            continue
        if set(e.get("biological_replicates", [])) != all_reps:
            continue
        peaks = details.get("reproducible_peaks")
        if peaks is None:
            continue
        entry_type = classify_repro_entry(details)
        candidates.append((float(peaks), entry_type))

    if not candidates:
        return None, None, 0

    max_peaks = max(c[0] for c in candidates)
    # Label based on which entry provided the max
    max_types = [t for p, t in candidates if p == max_peaks]

    if "optimal" in max_types:
        # Check if this came from IDR or overlap pipeline
        has_idr = any(
            t in ("optimal", "conservative")
            and "pr1_vs" in str(candidates)  # rough check
            for _, t in candidates
        )
        label = "optimal_idr" if has_idr else "optimal_overlap"
    elif "conservative" in max_types:
        label = "conservative_max"  # violation case
    else:
        label = "max_fallback"

    return max_peaks, label, len(candidates)

In [ ]:
def collect_experiment_metrics_complex(qc_metrics):
    """
    Build a per-experiment QC table from ENCODE quality metric objects.

    ENCODE provides multiple QC metric entries per experiment, computed at
    different pipeline stages and on different files. This function extracts
    a curated set of fields (defined by the module-level FIELDS_BY_TYPE dict),
    handles the distinction between replicate-level and experiment-level
    metrics, and aggregates into one row per experiment.

    Metric level distinction
    ------------------------
    ENCODE QC metrics fall into two categories:

    **Replicate-level** (ChipAlignmentQualityMetric, ChipAlignmentEnrichment-
    QualityMetric, ChipLibraryQualityMetric, ChipSeqFilterQualityMetric):
    computed per replicate BAM file. An experiment with 3 replicates will
    have 3 entries for each of these metric types. These are averaged across
    replicates, which is standard practice — the final bigwig is derived
    from all replicates, so the mean replicate-level QC characterizes the
    pooled output.

    **Experiment-level** (ChipPeakEnrichmentQualityMetric, ChipReplication-
    QualityMetric, HistoneChipSeqQualityMetric): computed on pooled data
    (peak calls from merged replicates). However, ENCODE also stores
    per-replicate peak metrics (e.g., FRiP for each replicate's peaks),
    and experiments reprocessed by multiple pipeline versions have separate
    pooled entries for each version. To avoid mixing per-replicate with
    pooled values, this function keeps only entries whose biological_replicates
    field matches the full set of replicates for the experiment (i.e., the
    pooled entry). For single-replicate experiments, all entries have
    biological_replicates=[1], so they pass through correctly. When multiple
    pipeline versions both produce pooled entries, their values are averaged;
    the spread is typically negligible.

    Aggregation
    -----------
    For each (metric_type, field) pair within an experiment, the function
    stores:
    - ``__mean``: mean across all qualifying entries
    - ``__n``: count of non-NaN entries (useful for diagnosing unexpected
      averaging — n=1 for experiment-level metrics in single-pipeline
      experiments, n=2-3 for replicate-level metrics)
    - ``__min``, ``__max``: range, only when n > 1, for verifying that
      averaged values are consistent

    Parameters
    ----------
    qc_metrics : dict[str, list[dict]]
        Mapping from experiment accession to a list of metric entries as
        returned by the ENCODE API. Each entry is a dict with keys:
        - 'quality_metric': dict containing '@type' (list of type strings),
          metric field names and values, and 'quality_metric_of' (list of
          file paths this metric applies to).
        - 'biological_replicates': list of int, indicating which biological
          replicates this metric covers (e.g. [1] for replicate 1, [1,2,3]
          for the pooled analysis across all replicates).
        - 'files': list of file paths this entry is associated with.

    Returns
    -------
    pd.DataFrame
        One row per experiment, indexed by experiment accession. Columns
        follow the naming convention ``<MetricType>__<field>__<stat>``
        where stat is one of mean, n, min, max. Only experiments with at
        least one extractable metric value are included.

    See Also
    --------
    FIELDS_BY_TYPE : module-level dict defining which fields to extract
        from each metric type.
    add_consensus_columns : coalesces the per-metric-type columns into
        short-named consensus columns (e.g. 'frip', 'NSC', 'NRF').
    """
    # raw[acc][(type, field)] = dict of step_run -> numeric value across qualifying entries
    raw = defaultdict(lambda: defaultdict(dict))

    # Experiment-level metric types: these have separate per-replicate and
    # pooled entries, and we only want the pooled ones.
    EXPERIMENT_LEVEL_TYPES = {
        "ChipPeakEnrichmentQualityMetric",
        "ChipReplicationQualityMetric",
        "HistoneChipSeqQualityMetric",
    }

    repro_peaks_audit = {}

    for acc, entries in qc_metrics.items():
        all_reps = set()
        for entry in entries:
            all_reps.update(entry.get("biological_replicates", []))

        # Special handling for reproducible_peaks
        peaks, peak_label, n_cand = select_reproducible_peaks(entries, all_reps)
        if peaks is not None:
            mtype = "ChipReplicationQualityMetric"
            field = "reproducible_peaks"
            # Store as single-entry dict to match expected format
            raw[acc][(mtype, field)]["selected"] = peaks
            repro_peaks_audit[acc] = {
                "peaks": peaks,
                "label": peak_label,
                "n_candidates": n_cand,
            }

        # All other metrics: existing logic
        for entry in entries:
            details = entry["quality_metric"]
            mtype = details["@type"][0]

            if mtype not in FIELDS_BY_TYPE:
                continue

            # Skip ChipReplicationQualityMetric — handled above
            if mtype == "ChipReplicationQualityMetric":
                continue

            bio_reps = entry.get("biological_replicates", [])
            if mtype in EXPERIMENT_LEVEL_TYPES and set(bio_reps) != all_reps:
                continue

            for field in FIELDS_BY_TYPE[mtype]:
                val = details.get(field)
                if val is None:
                    continue
                try:
                    step_run = details.get("step_run", id(details))
                    raw[acc][(mtype, field)][step_run] = float(val)
                except (TypeError, ValueError):
                    continue

    rows = []
    for acc, metric_dict in raw.items():
        row = {"experiment": acc}
        for (mtype, field), step_values in metric_dict.items():
            arr = np.asarray(list(step_values.values()), dtype=float)
            prefix = f"{mtype}__{field}"
            row[f"{prefix}__mean"] = float(np.nanmean(arr))
            row[f"{prefix}__n"] = int(np.sum(~np.isnan(arr)))
            if arr.size > 1:
                row[f"{prefix}__min"] = float(np.nanmin(arr))
                row[f"{prefix}__max"] = float(np.nanmax(arr))
        rows.append(row)

    df = pd.DataFrame(rows).set_index("experiment")

    # Attach audit info
    audit_df = pd.DataFrame.from_dict(repro_peaks_audit, orient="index")
    audit_df.index.name = "experiment"
    if not audit_df.empty:
        audit_df.columns = [
            "repro_peaks_selected",
            "repro_peaks_selection_type",
            "repro_peaks_n_candidates",
        ]
        df = df.join(audit_df)

    return df

In [ ]:
def verify_current_averaging_impact(qc_metrics, df_full):
    """
    Quantify the impact of the current averaging on reproducible_peaks.
    Compare the current mean to what you'd get using only optimal peaks.
    """
    optimal_vals = {}
    current_means = {}

    for acc, entries in qc_metrics.items():
        all_reps = set()
        for e in entries:
            all_reps.update(e.get("biological_replicates", []))

        pooled_peaks = []
        for e in entries:
            details = e["quality_metric"]
            if details["@type"][0] != "ChipReplicationQualityMetric":
                continue
            if set(e.get("biological_replicates", [])) != all_reps:
                continue
            peaks = details.get("reproducible_peaks")
            if peaks is None:
                continue
            pooled_peaks.append(float(peaks))

        if pooled_peaks:
            current_means[acc] = np.mean(pooled_peaks)
            # Optimal = max (optimal >= conservative by construction)
            optimal_vals[acc] = max(pooled_peaks)

    # Compare
    common = set(current_means) & set(optimal_vals) & set(df_full.index)

    current = np.array([current_means[a] for a in common])
    optimal = np.array([optimal_vals[a] for a in common])

    diff = optimal - current
    rel_diff = diff / np.where(current > 0, current, 1)

    changed = np.sum(np.abs(diff) > 0.5)  # float comparison threshold

    print(f"Experiments compared: {len(common)}")
    print(f"Experiments where optimal != current mean: {changed}")
    print("\nAbsolute difference (optimal - mean) for changed experiments:")
    print(pd.Series(diff[np.abs(diff) > 0.5]).describe())
    print("\nRelative difference for changed experiments:")
    print(pd.Series(rel_diff[np.abs(diff) > 0.5]).describe())

    # Spearman correlation between the two approaches
    rho, p = spearmanr(current, optimal)
    print(f"\nSpearman correlation between mean vs optimal: rho={rho:.4f}, p={p:.2e}")

    return pd.DataFrame(
        {
            "experiment": list(common),
            "current_mean": current,
            "optimal_max": optimal,
            "abs_diff": diff,
            "rel_diff": rel_diff,
        }
    ).set_index("experiment")

In [ ]:
df_full_new = collect_experiment_metrics_complex(qc_metrics)

# Audit the selection
print("Reproducible peaks selection type distribution:")
print(df_full_new["repro_peaks_selection_type"].value_counts(dropna=False).to_string())

# Verify: n=1 means no averaging happened
print("\nreproducible_peaks __n distribution:")
print(
    df_full_new["ChipReplicationQualityMetric__reproducible_peaks__n"]
    .value_counts()
    .to_string()
)

# Confirm spread is now zero
repro_col = "ChipReplicationQualityMetric__reproducible_peaks"
col_n = f"{repro_col}__n"
has_multiple = (
    df_full_new[df_full_new[col_n] > 1]
    if col_n in df_full_new.columns
    else pd.DataFrame()
)
print(f"\nExperiments with n > 1 for reproducible_peaks: {len(has_multiple)}")

Ignoring metrics with the "in progress status"

In [ ]:
status_counts = Counter()
for acc, entries in qc_metrics.items():
    for entry in entries:
        status = entry["quality_metric"].get("status", "unknown")
        status_counts[status] += 1
print(status_counts)

In [ ]:
def filter_released_metrics(qc_metrics):
    """Remove QC metric entries with status != 'released'."""
    filtered = {}
    for acc, entries in qc_metrics.items():
        released = [e for e in entries if e["quality_metric"].get("status") == "released"]
        if released:
            filtered[acc] = released
    return filtered


# qc_metrics = filter_released_metrics(qc_metrics)
# print(f"QC metrics after status filter: {len(qc_metrics)}")

In [ ]:
for col in remerged_df.columns:
    print(col)

In [ ]:
remerged_df = remerged_df.set_index("FILE_experiment_accession")

Impact of different treatment on repro peaks

In [ ]:
verify_current_averaging_impact(qc_metrics, remerged_df)

#### Metric extraction verification

The following cells verify that experiment-level metrics (FRiP, reproducible peaks) use only pooled entries and that the spread from averaging across pipeline versions is negligible. See `collect_experiment_metrics` docstring for details on the replicate-level vs experiment-level distinction.

Conclusion: spread is not negligible for repro. peaks, but using a more complex methodology did not have an effect on the global interpretation of results.

##### Metrics spread

In [ ]:
# Collect relative spread data for all metrics
spread_data = {}
for metric in ["frip", "reproducible_peaks", "jsd", "NSC"]:
    metric_lower = metric.lower()
    n_cols = [
        c for c in df_full.columns if metric_lower in c.lower() and c.endswith("__n")
    ]
    for col_n in n_cols:
        prefix = col_n.rsplit("__n", 1)[0]
        col_mean = f"{prefix}__mean"
        col_min = f"{prefix}__min"
        col_max = f"{prefix}__max"

        if col_mean not in df_full.columns:
            continue

        has_multiple = df_full[df_full[col_n] > 1]
        if len(has_multiple) == 0:
            continue

        spread = has_multiple[col_max] - has_multiple[col_min]
        iqr = df_full[col_mean].quantile(0.75) - df_full[col_mean].quantile(0.25)
        rel_spread = spread / iqr if iqr > 0 else spread

        # Short label: extract the metric type and field
        parts = prefix.split("__")
        label = f"{parts[0].replace('QualityMetric', '')}<br>{parts[1]}"

        spread_data[label] = rel_spread.dropna()

n_metrics = len(spread_data)
colors = [
    "rgba(31, 119, 180, 0.7)",
    "rgba(44, 160, 44, 0.7)",
    "rgba(214, 39, 40, 0.7)",
    "rgba(148, 103, 189, 0.7)",
    "rgba(255, 127, 14, 0.7)",
    "rgba(140, 86, 75, 0.7)",
]

fig = go.Figure()

for i, (label, values) in enumerate(spread_data.items()):
    color = colors[i % len(colors)]
    n = len(values)
    median = values.median()
    pct95 = values.quantile(0.95)

    fig.add_trace(
        go.Violin(
            y=values,
            x=[label] * len(values),
            name=f"{label} (n={n})",
            box_visible=True,
            meanline_visible=True,
            points="all",
            pointpos=-0.8,
            jitter=0.4,
            marker=dict(color=color, size=2, opacity=0.3),
            line_color="black",
            fillcolor=color.replace("0.7", "0.2"),
            showlegend=False,
            hovertemplate="relative spread: %{y:.4f}<extra></extra>",
            spanmode="hard",
        )
    )

    # Annotate median and 95th percentile
    fig.add_annotation(
        x=label,
        y=pct95,
        text=f"95th: {pct95:.3f}",
        showarrow=False,
        yshift=12,
        font=dict(size=9, color=color.replace("0.7", "1.0")),
    )


fig.update_yaxes(range=[0, 2])

# Reference lines
fig.add_hline(
    y=0.05,
    line_dash="dash",
    line_color="green",
    line_width=1,
    annotation_text="negligible (0.05)",
    annotation_position="top right",
    annotation_font_size=10,
    annotation_font_color="green",
)
fig.add_hline(
    y=0.15,
    line_dash="dash",
    line_color="orange",
    line_width=1,
    annotation_text="noticeable (0.15)",
    annotation_position="top right",
    annotation_font_size=10,
    annotation_font_color="orange",
)

fig.update_layout(
    title="Within-experiment spread of averaged QC metrics<br>"
    "<sup>Relative to between-experiment IQR</sup>",
    yaxis_title="Relative spread (fraction of IQR)",
    plot_bgcolor="white",
    width=900,
    height=600,
    yaxis=dict(
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=True,
        zerolinecolor="black",
        zerolinewidth=1,
    ),
    margin=dict(t=100, b=120),
)

fig.show()

In [ ]:
repro_col = "ChipReplicationQualityMetric__reproducible_peaks"
col_n = f"{repro_col}__n"
col_mean = f"{repro_col}__mean"
col_min = f"{repro_col}__min"
col_max = f"{repro_col}__max"

iqr = df_full[col_mean].quantile(0.75) - df_full[col_mean].quantile(0.25)

high_spread = df_full[df_full[col_n] > 1].copy()
high_spread["spread"] = high_spread[col_max] - high_spread[col_min]
high_spread["rel_spread"] = high_spread["spread"] / iqr

# Top 20 worst offenders
worst = high_spread.nlargest(20, "rel_spread")
print(worst[[col_mean, col_n, col_min, col_max, "spread", "rel_spread"]].to_string())

# Now look at the raw QC entries for the worst one
worst_acc = worst.index[0]
print(f"\n\nRaw entries for {worst_acc}:")
for entry in qc_metrics[worst_acc]:
    mtype = entry["quality_metric"]["@type"][0]
    if "Replication" not in mtype:
        continue
    reps = entry.get("biological_replicates", [])
    peaks = entry["quality_metric"].get("reproducible_peaks")
    files = entry.get("files", [])
    print(f"  bio_reps={reps}  peaks={peaks}  files={files}")

In [ ]:
df_full.loc[worst_acc, :]

In [ ]:
problematic_experiments = []
for acc in list(qc_metrics.keys()):
    entries = qc_metrics[acc]
    pooled_peaks = []
    all_reps = set()
    for e in entries:
        all_reps.update(e.get("biological_replicates", []))
    for e in entries:
        if e["quality_metric"]["@type"][0] != "ChipReplicationQualityMetric":
            continue
        if set(e.get("biological_replicates", [])) != all_reps:
            continue
        peaks = e["quality_metric"].get("reproducible_peaks")
        step = e["quality_metric"].get("step_run", "?")
        if peaks:
            pooled_peaks.append((peaks, step))
    if len(set(p for p, _ in pooled_peaks)) > 1:
        # print(f"{acc}: {pooled_peaks}")
        problematic_experiments.append((acc, pooled_peaks))